# Chapter 17 — Probability and Statistics by Simulation

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch17_statistics_by_simulation.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 17.1 Random numbers you can reproduce

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from indiadata import read_plfs, labour_status, labour_rates

rng = np.random.default_rng(seed=17)       # the generator used in this chapter
print("Ten rolls of a die:", rng.integers(1, 7, size=10))

a1 = np.random.default_rng(seed=2024).integers(1, 7, size=5)
a2 = np.random.default_rng(seed=2024).integers(1, 7, size=5)
print("Same seed, same numbers:", a1, a2)

In [ ]:
for n in [10, 100, 100_000]:
    tosses = rng.integers(0, 2, size=n)        # 1 = heads, 0 = tails
    print(f"{n:>7,} tosses: share of heads = {tosses.mean():.3f}")

## 17.2 Random variables and their distributions

In [ ]:
p_f = 0.417
X = stats.binom(n=10, p=p_f)
print(f"Mean {X.mean():.2f}, standard deviation {X.std():.2f}")
print(f"P(none of the 10 in the labour force) = {X.pmf(0):.3f}")
print(f"P(at most 3)                          = {X.cdf(3):.3f}")
print(f"P(more than half)                     = {1 - X.cdf(5):.3f}")

In [ ]:
draws = rng.binomial(n=10, p=p_f, size=100_000)
sim = pd.Series(draws).value_counts(normalize=True).sort_index()
pd.DataFrame({"formula": X.pmf(sim.index), "simulation": sim}).round(3).T

In [ ]:
Z = stats.norm(loc=0, scale=1)
print(f"P(-1.96 < Z < 1.96) = {Z.cdf(1.96) - Z.cdf(-1.96):.3f}")
print(f"The 97.5th percentile of Z is {Z.ppf(0.975):.3f}")
print(f"The 97.5th percentile of t with 9 d.f. is {stats.t(df=9).ppf(0.975):.3f}")

## 17.3 A population we can see

In [ ]:
cols = ["sex", "age", "pas", "sas", "acws", "ern_reg"]
p = read_plfs("perv1", columns=cols)

pop = p[(p["acws"] == 31) & (p["ern_reg"] > 0)
        & p["age"].between(15, 59) & p["sex"].isin([1, 2])]
earn = pop["ern_reg"].to_numpy(dtype=float)     # a plain NumPy array, ₹/month
female = (pop["sex"] == 2).to_numpy()
N = len(earn)

mu, sigma = earn.mean(), earn.std()
print(f"Population size N = {N:,} salaried workers")
print(f"Mean μ = ₹{mu:,.0f}   median = ₹{np.median(earn):,.0f}   "
      f"standard deviation σ = ₹{sigma:,.0f}")
print(f"Skewness = {stats.skew(earn):.2f}  (0 for a symmetric distribution)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(earn[earn <= 100_000] / 1000, bins=50, color="#9db7d5")
axes[0].axvline(mu / 1000, color="#c55a11", lw=1.5)
axes[0].text(mu / 1000 + 2, axes[0].get_ylim()[1] * 0.9,
             f"mean ₹{mu / 1000:.1f}k", color="#c55a11", fontsize=9)
axes[0].set_xlabel("Monthly earnings, ₹ thousand (shown up to ₹1 lakh)")
axes[0].set_title("Earnings: skewed", loc="left", fontsize=12)
axes[1].hist(np.log(earn), bins=50, color="#1f4e79")
axes[1].set_xlabel("log of monthly earnings")
axes[1].set_title("Log earnings: nearly symmetric", loc="left", fontsize=12)
for ax in axes:
    ax.set_ylabel("Workers")
    ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, f"Regular wage/salaried workers aged 15–59 (CWS), "
         f"unweighted records, N = {N:,}. Source: author's calculation from "
         "PLFS 2023–24 unit-level data (NSO, MoSPI).", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

## 17.4 The sampling distribution of a mean

In [ ]:
sample = rng.choice(earn, size=50, replace=False)
print(f"One sample of 50: mean = ₹{sample.mean():,.0f}  (true μ = ₹{mu:,.0f})")

In [ ]:
reps = 5_000
sizes = [10, 50, 400]
means = {}
for n in sizes:
    idx = rng.integers(0, N, size=(reps, n))
    means[n] = earn[idx].mean(axis=1)            # one mean per sample

table = pd.DataFrame({
    "average of the sample means": [means[n].mean() for n in sizes],
    "SD of the sample means": [means[n].std() for n in sizes],
    "theory: σ/√n": [sigma / np.sqrt(n) for n in sizes],
    "skewness of the sample means": [stats.skew(means[n]) for n in sizes],
}, index=pd.Index(sizes, name="n")).round(2)
table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, n in zip(axes, sizes):
    m = means[n] / 1000
    ax.hist(m, bins=60, density=True, color="#9db7d5")
    grid = np.linspace(m.min(), m.max(), 200)
    ax.plot(grid, stats.norm.pdf(grid, mu / 1000, sigma / np.sqrt(n) / 1000),
            color="#c55a11", lw=1.5)
    ax.axvline(mu / 1000, color="black", lw=0.8, ls="--")
    ax.set_title(f"n = {n}", loc="left", fontsize=12)
    ax.set_xlabel("Sample mean, ₹ thousand")
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)
fig.suptitle("The central limit theorem: 5,000 sample means of salaried earnings",
             x=0.01, ha="left", fontsize=13)
fig.text(0.01, -0.07, "Bars: simulated sample means. Orange line: the normal "
         "distribution N(μ, σ²/n) that the CLT predicts. Dashed line: the "
         "population mean μ.\nSource: author's simulation from PLFS 2023–24 "
         "unit-level data (NSO, MoSPI).", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

## 17.5 The law of large numbers

In [ ]:
stream = rng.choice(earn, size=20_000)
running = np.cumsum(stream) / np.arange(1, len(stream) + 1)

for n in [10, 100, 1_000, 20_000]:
    print(f"after {n:>6,} draws: running mean ₹{running[n - 1]:,.0f}  "
          f"(error {running[n - 1] - mu:+,.0f})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(np.arange(1, len(running) + 1), running / 1000, color="#1f4e79", lw=1.2)
ax.axhline(mu / 1000, color="#c55a11", ls="--", lw=1)
ax.text(1500, mu / 1000 + 0.35, f"population mean μ = ₹{mu / 1000:.1f}k",
        color="#c55a11", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Number of workers drawn (log scale)")
ax.set_ylabel("Running mean, ₹ thousand")
ax.set_title("The law of large numbers", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Source: author's simulation from PLFS 2023–24 unit-level "
         "data (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

## 17.6 The sampling distribution of a proportion

In [ ]:
p["status"] = labour_status(p["pas"], p["sas"])
adults = p[p["age"] >= 15]
in_lf = (adults["status"] != "out of LF").to_numpy(dtype=float)
p_true = in_lf.mean()
print(f"'Population' share in the labour force (unweighted records): {p_true:.3f}")

n = 500
p_hats = in_lf[rng.integers(0, len(in_lf), size=(reps, n))].mean(axis=1)
print(f"n = {n}: SD of 5,000 simulated p-hats = {p_hats.std():.4f}; "
      f"formula sqrt(p(1-p)/n) = {np.sqrt(p_true * (1 - p_true) / n):.4f}")

## 17.7 Confidence intervals, and do they really cover 95 per cent?

In [ ]:
def coverage(pop_values, n, reps=5_000, level=0.95):
    """Share of t-intervals from random samples that contain the true mean."""
    true = pop_values.mean()
    s = pop_values[rng.integers(0, len(pop_values), size=(reps, n))]
    m, se = s.mean(axis=1), s.std(axis=1, ddof=1) / np.sqrt(n)
    t = stats.t(df=n - 1).ppf(0.5 + level / 2)
    return np.mean((m - t * se <= true) & (true <= m + t * se))


cov = pd.DataFrame({
    "earnings (skewed)": [coverage(earn, n) for n in [10, 30, 100, 400]],
    "log earnings": [coverage(np.log(earn), n) for n in [10, 30, 100, 400]],
}, index=pd.Index([10, 30, 100, 400], name="n"))
(100 * cov).round(1)

In [ ]:
n, k = 30, 40
s = earn[rng.integers(0, N, size=(k, n))]
m, half = s.mean(axis=1), stats.t(n - 1).ppf(0.975) * s.std(axis=1, ddof=1) / np.sqrt(n)
hit = (m - half <= mu) & (mu <= m + half)

fig, ax = plt.subplots(figsize=(9, 4.2))
for i in range(k):
    colour = "#1f4e79" if hit[i] else "#c55a11"
    ax.plot([i, i], [(m[i] - half[i]) / 1000, (m[i] + half[i]) / 1000],
            color=colour, lw=2)
    ax.plot(i, m[i] / 1000, "o", color=colour, ms=3)
ax.axhline(mu / 1000, color="black", ls="--", lw=1)
ax.set_xlabel("Sample number")
ax.set_ylabel("95% interval for μ, ₹ thousand")
ax.set_title(f"40 confidence intervals from samples of n = {n}: "
             f"{(~hit).sum()} miss μ (orange)", loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.04, "Dashed line: population mean μ. Source: author's simulation "
         "from PLFS 2023–24 unit-level data (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

## 17.8 Testing a hypothesis: is there a gender earnings gap?

In [ ]:
gap_true = earn[~female].mean() - earn[female].mean()
print(f"Men ₹{earn[~female].mean():,.0f}, women ₹{earn[female].mean():,.0f}: "
      f"gap ₹{gap_true:,.0f}")

In [ ]:
men_pop, women_pop = earn[~female], earn[female]
ym = rng.choice(men_pop, size=100, replace=False)
yf = rng.choice(women_pop, size=100, replace=False)

t_hand = (ym.mean() - yf.mean()) / np.sqrt(ym.var(ddof=1) / 100 + yf.var(ddof=1) / 100)
res = stats.ttest_ind(ym, yf, equal_var=False)
print(f"Sample gap: ₹{ym.mean() - yf.mean():,.0f}")
print(f"t by hand = {t_hand:.3f};  scipy t = {res.statistic:.3f}, "
      f"p-value = {res.pvalue:.4f}")

In [ ]:
women = adults[adults["sex"] == 2]
lf_w = (women["status"] != "out of LF").to_numpy(dtype=float)
rural = (women["sec"] == 1).to_numpy()
print(f"Population LFPR: rural women {lf_w[rural].mean():.3f}, "
      f"urban women {lf_w[~rural].mean():.3f}")

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

r = rng.choice(lf_w[rural], size=300, replace=False)
u = rng.choice(lf_w[~rural], size=300, replace=False)
z, pval = proportions_ztest(count=[r.sum(), u.sum()], nobs=[300, 300])
print(f"Sample LFPR rural {r.mean():.3f}, urban {u.mean():.3f}; "
      f"z = {z:.2f}, p-value = {pval:.2g}")

## 17.9 Type I error and power, by simulation

In [ ]:
def welch_reject(pop_a, pop_b, n, reps=4_000, alpha=0.05):
    """Share of simulated Welch t-tests (n per group) that reject H0 at alpha."""
    a = pop_a[rng.integers(0, len(pop_a), size=(reps, n))]
    b = pop_b[rng.integers(0, len(pop_b), size=(reps, n))]
    t = (a.mean(1) - b.mean(1)) / np.sqrt(a.var(1, ddof=1) / n + b.var(1, ddof=1) / n)
    df = n - 1                                     # a safe, conservative choice
    return np.mean(np.abs(t) > stats.t(df).ppf(1 - alpha / 2))


print(f"H0 true (men vs men), n = 100 each: rejection rate = "
      f"{welch_reject(men_pop, men_pop, 100):.3f}")

In [ ]:
ns = [10, 20, 40, 80, 160, 320]
power = pd.Series([welch_reject(men_pop, women_pop, n) for n in ns],
                  index=pd.Index(ns, name="n per group"), name="power")
power.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(power.index, power.values, marker="o", color="#1f4e79")
ax.axhline(0.8, color="grey", ls=":", lw=1)
ax.text(ns[0], 0.82, "conventional target: 80% power", fontsize=9, color="grey")
ax.axhline(0.05, color="#c55a11", ls="--", lw=1)
ax.text(160, 0.08, "size of the test: 5%", fontsize=9, color="#c55a11")
ax.set_xscale("log", base=2)
ax.set_xticks(ns)
ax.set_xticklabels(ns)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Workers sampled in each group (log scale)")
ax.set_ylabel("Probability of rejecting H₀")
ax.set_title("Power of a t-test to detect the gender earnings gap",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Welch t-test at the 5% level, 4,000 simulated surveys "
         "per point. Source: author's simulation from PLFS 2023–24 unit-level "
         "data (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

## 17.10 The bootstrap: standard errors without a formula

In [ ]:
n = 200
y = rng.choice(earn, size=n, replace=False)           # our one real sample
boot = np.median(y[rng.integers(0, n, size=(2_000, n))], axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Sample median ₹{np.median(y):,.0f}; bootstrap SE ₹{boot.std():,.0f}")
print(f"95% percentile interval: ₹{lo:,.0f} to ₹{hi:,.0f}")

In [ ]:
true_sd = np.median(earn[rng.integers(0, N, size=(2_000, n))], axis=1).std()
print(f"True SD of the sample median (n = {n}): ₹{true_sd:,.0f}")
print(f"Population median ₹{np.median(earn):,.0f}")

## 17.11 Back to the real survey: clusters, weights and design effects

In [ ]:
adults = adults.assign(
    lf=(adults["status"] != "out of LF").astype(float),
    woman=(adults["sex"] == 2).astype(float),
    stratum=adults.groupby(["st", "sec", "strm", "sstrm"]).ngroup())
print(f"Female LFPR, weighted: {labour_rates(adults[adults['sex'] == 2])['LFPR']:.1f}%")

In [ ]:
wom = adults[adults["sex"] == 2]
w = wom["weight"].to_numpy(dtype=float)
lf = wom["lf"].to_numpy()
phat = np.average(lf, weights=w)
n_w = len(wom)
n_eff = w.sum() ** 2 / (w ** 2).sum()

se_srs = np.sqrt(phat * (1 - phat) / n_w)
se_kish = np.sqrt(phat * (1 - phat) / n_eff)
print(f"Women in the sample: {n_w:,}; Kish effective sample size: {n_eff:,.0f}")
print(f"SE, SRS formula: {100 * se_srs:.3f} points; with weights (Kish): "
      f"{100 * se_kish:.3f} points")

In [ ]:
def fsu_totals(df, y, x):
    """Weighted totals of y and x for every FSU, with its stratum."""
    d = df.assign(wy=df["weight"] * df[y], wx=df["weight"] * df[x])
    return d.groupby("mfsu").agg(stratum=("stratum", "first"),
                                 Y=("wy", "sum"), X=("wx", "sum"))


def linearised_se(t):
    """Stratified, clustered (Taylor linearisation) SE of R = sum Y / sum X."""
    R = t["Y"].sum() / t["X"].sum()
    z = (t["Y"] - R * t["X"]) / t["X"].sum()
    g = z.groupby(t["stratum"])
    nh = g.transform("size")
    dev = z - g.transform("mean")
    return R, np.sqrt((nh / (nh - 1) * dev ** 2).sum())


adults["lf_woman"] = adults["lf"] * adults["woman"]
tot = fsu_totals(adults, "lf_woman", "woman")
R, se_lin = linearised_se(tot)
print(f"{len(tot):,} FSUs in {tot['stratum'].nunique()} strata")
print(f"LFPR {100 * R:.2f}%, linearised SE {100 * se_lin:.3f} points")

In [ ]:
def rao_wu_factors(strata, reps):
    """Replicate weight factors (reps x FSUs) for the Rao-Wu bootstrap."""
    strata = np.asarray(strata)
    order = np.argsort(strata, kind="stable")
    s_sorted = strata[order]
    starts = np.flatnonzero(np.r_[True, s_sorted[1:] != s_sorted[:-1]])
    sizes = np.diff(np.r_[starts, len(s_sorted)])
    slot_start = np.repeat(starts, sizes - 1)       # n_h - 1 draws per stratum
    slot_size = np.repeat(sizes, sizes - 1)
    F = np.zeros((reps, len(strata)))
    for r in range(reps):
        pick = order[slot_start + rng.integers(0, slot_size)]
        F[r] = np.bincount(pick, minlength=len(strata))
    scale = np.empty(len(strata))
    scale[order] = np.repeat(sizes / (sizes - 1), sizes)
    return F * scale


F = rao_wu_factors(tot["stratum"], reps=500)
R_boot = (F @ tot["Y"].to_numpy()) / (F @ tot["X"].to_numpy())
se_boot = R_boot.std(ddof=1)
print(f"Rao–Wu bootstrap SE ({len(R_boot)} replicates): {100 * se_boot:.3f} points")

In [ ]:
sub = wom.assign(w_sub=wom["mult"] / 100 / wom["no_qtr"])
est = [np.average(sub.loc[sub["ss"] == k, "lf"], weights=sub.loc[sub["ss"] == k, "w_sub"])
       for k in (1, 2)]
se_sub = abs(est[0] - est[1]) / 2
print(f"Sub-samples: {100 * est[0]:.2f}% and {100 * est[1]:.2f}%; "
      f"half-difference SE {100 * se_sub:.3f} points")

In [ ]:
def se_table(ses, scale=100, label="SE (percentage points)"):
    """SEs side by side, with each variance relative to the first (SRS) one."""
    t = pd.DataFrame({label: scale * pd.Series(ses)})
    t["design effect (variance ratio)"] = (t.iloc[:, 0] / t.iloc[0, 0]) ** 2
    return t.round(3)


lfpr_table = se_table({
    "1. SRS formula, unweighted": se_srs,
    "2. SRS with weights (Kish n_eff)": se_kish,
    "3. Stratified cluster, linearised": se_lin,
    "4. Stratified cluster, Rao–Wu bootstrap": se_boot,
    "5. Two sub-samples (Ch 14)": se_sub,
})
lfpr_table

In [ ]:
m_bar = wom.groupby("mfsu").size().mean()
deff_cluster = (se_lin / se_kish) ** 2          # clustering on top of weights
rho = (deff_cluster - 1) / (m_bar - 1)
print(f"Women per FSU: {m_bar:.1f}; deff from clustering: {deff_cluster:.2f}; "
      f"implied intra-cluster correlation ρ ≈ {rho:.2f}")

In [ ]:
adults["salaried"] = ((adults["acws"] == 31) & (adults["ern_reg"] > 0)
                      & (adults["age"] <= 59)).astype(float)
adults["earn_s"] = adults["ern_reg"].astype(float).fillna(0) * adults["salaried"]
tot_e = fsu_totals(adults, "earn_s", "salaried")
R_e, se_lin_e = linearised_se(tot_e)
F_e = rao_wu_factors(tot_e["stratum"], reps=500)
se_boot_e = ((F_e @ tot_e["Y"].to_numpy()) / (F_e @ tot_e["X"].to_numpy())).std(ddof=1)

sal = adults[adults["salaried"] == 1]
ws, ys = sal["weight"].to_numpy(dtype=float), sal["ern_reg"].to_numpy(dtype=float)
sd_w = np.sqrt(np.average((ys - R_e) ** 2, weights=ws))
print(f"Weighted mean salary: ₹{R_e:,.0f} per month, from {len(sal):,} workers")
earn_table = se_table({
    "1. SRS formula, unweighted": sd_w / np.sqrt(len(sal)),
    "2. SRS with weights (Kish n_eff)": sd_w / np.sqrt(ws.sum() ** 2 / (ws ** 2).sum()),
    "3. Stratified cluster, linearised": se_lin_e,
    "4. Stratified cluster, Rao–Wu bootstrap": se_boot_e,
}, scale=1, label="SE (₹)")
print(f"Extra factor from clustering: {(se_lin_e / earn_table.iloc[1, 0]) ** 2:.2f}")
earn_table

In [ ]:
del p, adults, wom, sub, sal, tot, tot_e, F, F_e      # free memory

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading